In [1]:
import gc
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

parquet_path = "/kaggle/input/datasets/tjaycuz/us-flight-data/US_flight_data.parquet"

pf = pq.ParquetFile(parquet_path)
print(f"Rows: {pf.metadata.num_rows:,} | Columns: {pf.metadata.num_columns}")

Rows: 32,957,871 | Columns: 119


In [2]:
md = pf.metadata
names = pf.schema_arrow.names
nulls = dict.fromkeys(names, 0)

for rg in range(md.num_row_groups):
    row_group = md.row_group(rg)
    for i, name in enumerate(names):
        stats = row_group.column(i).statistics
        if stats is not None and stats.has_null_count:
            nulls[name] += stats.null_count

null_df = (pd.DataFrame({"column": list(nulls), "nulls": list(nulls.values())})
             .assign(null_pct=lambda d: (d["nulls"] / md.num_rows * 100).round(2))
             .sort_values("null_pct", ascending=False)
             .reset_index(drop=True))
null_df.head(40)

,column,nulls,null_pct
0,SCH_OP_CARRIER_AIRLINE_ID,32956489,100.00
1,SCH_OP_CARRIER,32956489,100.00
2,SCH_OP_CARRIER_FL_NUM,32956489,100.00
3,SCH_OP_UNIQUE_CARRIER,32956489,100.00
4,DIV3_LONGEST_GTIME,32957862,100.00
5,DIV3_WHEELS_OFF,32957866,100.00
6,DIV3_TAIL_NUM,32957866,100.00
7,DIV4_AIRPORT,32957871,100.00
8,DIV4_AIRPORT_ID,32957871,100.00
9,DIV4_AIRPORT_SEQ_ID,32957871,100.00


In [2]:
names = pf.schema_arrow.names
md = pf.metadata
nulls = dict.fromkeys(names, 0)
for rg in range(md.num_row_groups):
    r = md.row_group(rg)
    for i, n in enumerate(names):
        nulls[n] += r.column(i).statistics.null_count

total = md.num_rows
for i, n in enumerate(names):
    print(f"{i:3d}  {n:<28} {nulls[n]/total*100:6.2f}% null")

  0  YEAR                           0.00% null
  1  QUARTER                        0.00% null
  2  MONTH                          0.00% null
  3  DAY_OF_MONTH                   0.00% null
  4  DAY_OF_WEEK                    0.00% null
  5  FL_DATE                        0.00% null
  6  MKT_UNIQUE_CARRIER            64.06% null
  7  BRANDED_CODE_SHARE            64.06% null
  8  MKT_CARRIER_AIRLINE_ID        64.06% null
  9  MKT_CARRIER                   64.06% null
 10  MKT_CARRIER_FL_NUM            64.06% null
 11  SCH_OP_UNIQUE_CARRIER        100.00% null
 12  SCH_OP_CARRIER_AIRLINE_ID    100.00% null
 13  SCH_OP_CARRIER               100.00% null
 14  SCH_OP_CARRIER_FL_NUM        100.00% null
 15  OP_UNIQUE_CARRIER              0.00% null
 16  OP_CARRIER_AIRLINE_ID          0.00% null
 17  OP_CARRIER                     0.00% null
 18  TAIL_NUM                       0.33% null
 19  OP_CARRIER_FL_NUM              0.00% null
 20  ORIGIN_AIRPORT_ID              0.00% null
 21  ORIGIN_A

In [3]:
drop_cols = [n for n in names if nulls[n] / total >= 0.99]
keep_cols = [n for n in names if n not in drop_cols]
print(f"Drop {len(drop_cols)} | Keep {len(keep_cols)}")
print(drop_cols)

Drop 51 | Keep 68
['SCH_OP_UNIQUE_CARRIER', 'SCH_OP_CARRIER_AIRLINE_ID', 'SCH_OP_CARRIER', 'SCH_OP_CARRIER_FL_NUM', 'FIRST_DEP_TIME', 'TOTAL_ADD_GTIME', 'LONGEST_ADD_GTIME', 'DIV_REACHED_DEST', 'DIV_ACTUAL_ELAPSED_TIME', 'DIV_ARR_DELAY', 'DIV_DISTANCE', 'DIV1_AIRPORT', 'DIV1_AIRPORT_ID', 'DIV1_AIRPORT_SEQ_ID', 'DIV1_WHEELS_ON', 'DIV1_TOTAL_GTIME', 'DIV1_LONGEST_GTIME', 'DIV1_WHEELS_OFF', 'DIV1_TAIL_NUM', 'DIV2_AIRPORT', 'DIV2_AIRPORT_ID', 'DIV2_AIRPORT_SEQ_ID', 'DIV2_WHEELS_ON', 'DIV2_TOTAL_GTIME', 'DIV2_LONGEST_GTIME', 'DIV2_WHEELS_OFF', 'DIV2_TAIL_NUM', 'DIV3_AIRPORT', 'DIV3_AIRPORT_ID', 'DIV3_AIRPORT_SEQ_ID', 'DIV3_WHEELS_ON', 'DIV3_TOTAL_GTIME', 'DIV3_LONGEST_GTIME', 'DIV3_WHEELS_OFF', 'DIV3_TAIL_NUM', 'DIV4_AIRPORT', 'DIV4_AIRPORT_ID', 'DIV4_AIRPORT_SEQ_ID', 'DIV4_WHEELS_ON', 'DIV4_TOTAL_GTIME', 'DIV4_LONGEST_GTIME', 'DIV4_WHEELS_OFF', 'DIV4_TAIL_NUM', 'DIV5_AIRPORT', 'DIV5_AIRPORT_ID', 'DIV5_AIRPORT_SEQ_ID', 'DIV5_WHEELS_ON', 'DIV5_TOTAL_GTIME', 'DIV5_LONGEST_GTIME', 'DIV5_WHEELS

In [4]:
import time
text_cols = ["MKT_UNIQUE_CARRIER","BRANDED_CODE_SHARE","MKT_CARRIER","OP_UNIQUE_CARRIER","OP_CARRIER",
             "TAIL_NUM","ORIGIN","ORIGIN_CITY_NAME","ORIGIN_STATE_ABR","ORIGIN_STATE_NM",
             "DEST","DEST_CITY_NAME","DEST_STATE_ABR","DEST_STATE_NM","FL_DATE",
             "DEP_TIME_BLK","ARR_TIME_BLK","CANCELLATION_CODE","DUP"]
text_cols = [c for c in text_cols if c in keep_cols]
out_path = "/kaggle/working/flights_typed.parquet"

writer, t0, n = None, time.time(), 0
for batch in pf.iter_batches(batch_size=1_000_000, columns=keep_cols):
    arrays = [batch.column(c) if c in text_cols else pc.cast(batch.column(c), pa.float32()) for c in keep_cols]
    tbl = pa.Table.from_arrays(arrays, names=keep_cols)
    if writer is None:
        writer = pq.ParquetWriter(out_path, tbl.schema, compression="snappy")
    writer.write_table(tbl)
    n += len(tbl); print(f"{n:,} rows | {time.time()-t0:.0f}s")
writer.close()

1,000,000 rows | 25s
2,000,000 rows | 49s
3,000,000 rows | 73s
4,000,000 rows | 98s
5,000,000 rows | 122s
6,000,000 rows | 145s
7,000,000 rows | 168s
8,000,000 rows | 191s
9,000,000 rows | 215s
10,000,000 rows | 238s
11,000,000 rows | 261s
12,000,000 rows | 284s
13,000,000 rows | 305s
14,000,000 rows | 324s
15,000,000 rows | 344s
16,000,000 rows | 361s
17,000,000 rows | 381s
18,000,000 rows | 402s
19,000,000 rows | 423s
20,000,000 rows | 443s
21,000,000 rows | 459s
22,000,000 rows | 481s
23,000,000 rows | 501s
24,000,000 rows | 522s
25,000,000 rows | 544s
26,000,000 rows | 562s
27,000,000 rows | 583s
28,000,000 rows | 604s
29,000,000 rows | 626s
30,000,000 rows | 646s
31,000,000 rows | 663s
32,000,000 rows | 685s
32,957,871 rows | 702s


In [1]:
import gc
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

In [ ]:
df = pq.read_table(out_path, read_dictionary=text_cols).to_pandas()

In [6]:
df.head()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,MKT_UNIQUE_CARRIER,BRANDED_CODE_SHARE,MKT_CARRIER_AIRLINE_ID,MKT_CARRIER,...,AIR_TIME,FLIGHTS,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_AIRPORT_LANDINGS
0,2022.0,1.0,1.0,1.0,6.0,1/1/2022 12:00:00 AM,AA,AA,19805.0,AA,...,347.0,1.0,2475.0,10.0,NaN,NaN,NaN,NaN,NaN,0.0
1,2022.0,1.0,1.0,1.0,6.0,1/1/2022 12:00:00 AM,AA,AA,19805.0,AA,...,270.0,1.0,2475.0,10.0,NaN,NaN,NaN,NaN,NaN,0.0
2,2022.0,1.0,1.0,1.0,6.0,1/1/2022 12:00:00 AM,AA,AA,19805.0,AA,...,NaN,1.0,258.0,2.0,NaN,NaN,NaN,NaN,NaN,0.0
3,2022.0,1.0,1.0,1.0,6.0,1/1/2022 12:00:00 AM,AA,AA,19805.0,AA,...,166.0,1.0,1182.0,5.0,907.0,0.0,3.0,0.0,0.0,0.0
4,2022.0,1.0,1.0,1.0,6.0,1/1/2022 12:00:00 AM,AA,AA,19805.0,AA,...,104.0,1.0,575.0,3.0,16.0,0.0,0.0,0.0,3.0,0.0


In [10]:
info = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "null_%": (df.isna().mean() * 100).round(2),
    "nunique": df.nunique(),
    "MB": (df.memory_usage(deep=True) / 1e6).round(0),
})
print(f"Total: {info['MB'].sum()/1e3:.1f} GB")
info

Total: 7.3 GB


,dtype,null_%,nunique,MB
ACTUAL_ELAPSED_TIME,float32,2.04,749.0,132.0
AIR_TIME,float32,2.04,701.0,132.0
ARR_DEL15,float32,2.04,2.0,132.0
ARR_DELAY,float32,2.04,2428.0,132.0
ARR_DELAY_GROUP,float32,2.04,15.0,132.0
...,...,...,...,...
TAXI_OUT,float32,1.77,217.0,132.0
WEATHER_DELAY,float32,78.94,1441.0,132.0
WHEELS_OFF,float32,1.77,1440.0,132.0
WHEELS_ON,float32,1.81,1440.0,132.0


In [13]:
num_cols = df.select_dtypes("number").columns
info["min"] = df[num_cols].min()
info["max"] = df[num_cols].max()
info["range"] = info["min"].astype(str) + "  to  " + info["max"].astype(str)
info.loc[info["min"].isna(), "range"] = ""            # text/category columns
info["whole_nums"] = pd.Series({c: bool((df[c].dropna() % 1 == 0).all()) for c in num_cols})

pd.set_option("display.max_rows", None, "display.width", 200)
print(info[["dtype", "null_%", "nunique", "range", "whole_nums", "MB"]].to_string())

                           dtype  null_%  nunique                     range whole_nums     MB
ACTUAL_ELAPSED_TIME      float32    2.04    749.0           14.0  to  965.0       True  132.0
AIR_TIME                 float32    2.04    701.0            5.0  to  953.0       True  132.0
ARR_DEL15                float32    2.04      2.0              0.0  to  1.0       True  132.0
ARR_DELAY                float32    2.04   2428.0        -128.0  to  7232.0       True  132.0
ARR_DELAY_GROUP          float32    2.04     15.0            -2.0  to  12.0       True  132.0
ARR_DELAY_NEW            float32    2.04   2322.0           0.0  to  7232.0       True  132.0
ARR_TIME                 float32    1.81   1440.0           1.0  to  2400.0       True  132.0
ARR_TIME_BLK            category    0.00     19.0                                  NaN   33.0
BRANDED_CODE_SHARE      category   64.06     14.0                                  NaN   33.0
CANCELLATION_CODE       category   98.22      4.0           

In [14]:
i8  = ["QUARTER","MONTH","DAY_OF_MONTH","DAY_OF_WEEK","CANCELLED","DIVERTED","DISTANCE_GROUP",
       "DIV_AIRPORT_LANDINGS","FLIGHTS","ORIGIN_STATE_FIPS","DEST_STATE_FIPS","ORIGIN_WAC","DEST_WAC"]
i16 = ["YEAR","CRS_DEP_TIME","CRS_ARR_TIME","CRS_ELAPSED_TIME","DISTANCE","OP_CARRIER_FL_NUM",
       "OP_CARRIER_AIRLINE_ID","ORIGIN_AIRPORT_ID","DEST_AIRPORT_ID"]
u16 = ["ORIGIN_CITY_MARKET_ID","DEST_CITY_MARKET_ID"]              # max 36,422 > int16 limit
i32 = ["ORIGIN_AIRPORT_SEQ_ID","DEST_AIRPORT_SEQ_ID"]              # ~1.6M
# columns with nulls -> pandas nullable ints
n8  = ["DEP_DEL15","DEP_DELAY_GROUP","ARR_DEL15","ARR_DELAY_GROUP"]
n16 = ["DEP_TIME","ARR_TIME","WHEELS_OFF","WHEELS_ON","TAXI_OUT","TAXI_IN","AIR_TIME","ACTUAL_ELAPSED_TIME",
       "DEP_DELAY_NEW","ARR_DELAY","ARR_DELAY_NEW","CARRIER_DELAY","WEATHER_DELAY","NAS_DELAY",
       "SECURITY_DELAY","LATE_AIRCRAFT_DELAY","MKT_CARRIER_FL_NUM"]
nu16 = ["MKT_CARRIER_AIRLINE_ID"]

plan = {**dict.fromkeys(i8,"int8"), **dict.fromkeys(i16,"int16"), **dict.fromkeys(u16,"uint16"),
        **dict.fromkeys(i32,"int32"), **dict.fromkeys(n8,"Int8"), **dict.fromkeys(n16,"Int16"),
        **dict.fromkeys(nu16,"UInt16")}

before = df.memory_usage(deep=True).sum()/1e9
for c, t in plan.items():
    df[c] = df[c].astype(t)
gc.collect()
print(f"{before:.1f} GB -> {df.memory_usage(deep=True).sum()/1e9:.1f} GB")
print(df.dtypes.value_counts())

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

In [15]:
bad = [c for c, t in plan.items() if t[0].islower() and df[c].isna().any()]
print("Columns with hidden nulls:", bad)
for c in bad:
    plan[c] = plan[c].replace("uint", "UInt").replace("int", "Int")   # int16 -> Int16, uint16 -> UInt16

before = df.memory_usage(deep=True).sum()/1e9
for c, t in plan.items():
    if str(df[c].dtype) != t:
        df[c] = df[c].astype(t)
gc.collect()
print(f"{before:.1f} GB -> {df.memory_usage(deep=True).sum()/1e9:.1f} GB")
print(df.dtypes.value_counts())

Columns with hidden nulls: ['DIV_AIRPORT_LANDINGS', 'CRS_ELAPSED_TIME', 'OP_CARRIER_FL_NUM']
6.6 GB -> 4.5 GB
Int16       19
int8        12
int16        7
Int8         5
category     2
int32        2
category     2
category     2
category     2
category     2
uint16       2
UInt16       1
category     1
category     1
category     1
category     1
category     1
category     1
float32      1
category     1
category     1
category     1
Name: count, dtype: int64


In [16]:
print(pd.crosstab(df["DEP_DELAY"].isna(), [df["CANCELLED"], df["DIVERTED"]], margins=True))

CANCELLED         0              1       All
DIVERTED          0      1       0          
DEP_DELAY                                   
False      32284613  86850   19447  32390910
True              0      0  566961    566961
All        32284613  86850  586408  32957871


In [18]:

df.dtypes

YEAR                         int16
QUARTER                       int8
MONTH                         int8
DAY_OF_MONTH                  int8
DAY_OF_WEEK                   int8
FL_DATE                   category
MKT_UNIQUE_CARRIER        category
BRANDED_CODE_SHARE        category
MKT_CARRIER_AIRLINE_ID      UInt16
MKT_CARRIER               category
MKT_CARRIER_FL_NUM           Int16
OP_UNIQUE_CARRIER         category
OP_CARRIER_AIRLINE_ID        int16
OP_CARRIER                category
TAIL_NUM                  category
OP_CARRIER_FL_NUM            Int16
ORIGIN_AIRPORT_ID            int16
ORIGIN_AIRPORT_SEQ_ID        int32
ORIGIN_CITY_MARKET_ID       uint16
ORIGIN                    category
ORIGIN_CITY_NAME          category
ORIGIN_STATE_ABR          category
ORIGIN_STATE_FIPS             int8
ORIGIN_STATE_NM           category
ORIGIN_WAC                    int8
DEST_AIRPORT_ID              int16
DEST_AIRPORT_SEQ_ID          int32
DEST_CITY_MARKET_ID         uint16
DEST                

In [19]:
df.to_parquet("/kaggle/working/flights_df_68cols.parquet", compression="snappy", index=False)
import os; print(f"{os.path.getsize('/kaggle/working/flights_df_68cols.parquet')/1e9:.2f} GB saved")

1.51 GB saved


In [20]:
from IPython.display import FileLink
FileLink("flights_df_68cols.parquet")

/kaggle/working/flights_df_68cols.parquet

----------------------------------------------------------------------------------------------------

In [1]:
import pandas as pd

path = "/kaggle/input/datasets/tjaycuz/68-col-v2/flights_df_68cols (1).parquet"
df = pd.read_parquet(path)
print(df.shape)

num = df.select_dtypes("number")
summary = pd.DataFrame({"dtype": df.dtypes.astype(str)})
summary["min"] = num.min()
summary["max"] = num.max()
summary["nulls"] = df.isna().sum()
pd.set_option("display.max_rows", None)
summary

(32957871, 68)


,dtype,min,max,nulls
YEAR,int16,2022.0,2026.0,0
QUARTER,int8,1.0,4.0,0
MONTH,int8,1.0,12.0,0
DAY_OF_MONTH,int8,1.0,31.0,0
DAY_OF_WEEK,int8,1.0,7.0,0
FL_DATE,category,<NA>,<NA>,0
MKT_UNIQUE_CARRIER,category,<NA>,<NA>,21111909
BRANDED_CODE_SHARE,category,<NA>,<NA>,21111909
MKT_CARRIER_AIRLINE_ID,UInt16,19393.0,20436.0,21111909
MKT_CARRIER,category,<NA>,<NA>,21111909


In [2]:
pd.set_option("display.max_rows", None)

summary

,dtype,min,max,nulls
YEAR,int16,2022.0,2026.0,0
QUARTER,int8,1.0,4.0,0
MONTH,int8,1.0,12.0,0
DAY_OF_MONTH,int8,1.0,31.0,0
DAY_OF_WEEK,int8,1.0,7.0,0
FL_DATE,category,<NA>,<NA>,0
MKT_UNIQUE_CARRIER,category,<NA>,<NA>,21111909
BRANDED_CODE_SHARE,category,<NA>,<NA>,21111909
MKT_CARRIER_AIRLINE_ID,UInt16,19393.0,20436.0,21111909
MKT_CARRIER,category,<NA>,<NA>,21111909


In [3]:
df["FL_DATE"] = pd.to_datetime(df["FL_DATE"].astype(str))
print(df["FL_DATE"].min(), df["FL_DATE"].max())

bad = df[df["CRS_ELAPSED_TIME"] <= 0]
print(len(bad))
bad[["FL_DATE","OP_UNIQUE_CARRIER","ORIGIN","DEST","CRS_DEP_TIME","CRS_ARR_TIME","CRS_ELAPSED_TIME","DISTANCE"]].head(20)

/tmp/ipykernel_49/1946666277.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["FL_DATE"] = pd.to_datetime(df["FL_DATE"].astype(str))


2022-01-01 00:00:00 2026-08-31 00:00:00
21


,FL_DATE,OP_UNIQUE_CARRIER,ORIGIN,DEST,CRS_DEP_TIME,CRS_ARR_TIME,CRS_ELAPSED_TIME,DISTANCE
378148,2022-01-21,YX,BNA,EWR,600,922,-48,748
5533880,2022-10-14,YX,IND,LGA,605,811,-44,660
5706328,2022-10-23,YX,PIT,BOS,825,804,-36,496
6539245,2022-12-06,YX,PIT,BOS,605,805,-85,496
8570162,2023-03-23,YX,JFK,PWM,2240,2235,-5,273
13827657,2023-12-14,OO,DFW,GCK,1353,921,-272,404
15724889,2024-03-27,WN,LAS,RNO,1840,1600,-160,345
27394292,2025-11-17,WN,LAS,BUR,1724,1545,-99,223
27394416,2025-11-17,WN,LAS,BUR,1802,1655,-67,223
27426411,2025-11-19,OO,DTW,LEX,1053,953,-60,296


----------------------------------------------------------------------------------------------------

### Project begins

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


In [2]:
path = "/kaggle/input/datasets/tjaycuz/68-col-v2/flights_df_68cols (1).parquet"
df = pd.read_parquet(path)




In [8]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 250)
pd.options.display.float_format = '{:,.2f}'.format


df.head()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,MKT_UNIQUE_CARRIER,BRANDED_CODE_SHARE,MKT_CARRIER_AIRLINE_ID,MKT_CARRIER,MKT_CARRIER_FL_NUM,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_FIPS,ORIGIN_STATE_NM,ORIGIN_WAC,DEST_AIRPORT_ID,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,DEST,DEST_CITY_NAME,DEST_STATE_ABR,DEST_STATE_FIPS,DEST_STATE_NM,DEST_WAC,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,DEP_DELAY_GROUP,DEP_TIME_BLK,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,ARR_DELAY_GROUP,ARR_TIME_BLK,CANCELLED,CANCELLATION_CODE,DIVERTED,DUP,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,FLIGHTS,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_AIRPORT_LANDINGS
0,2022,1,1,1,6,1/1/2022 12:00:00 AM,AA,AA,19805,AA,1,AA,19805,AA,N106NN,1,12478,1247805,31703,JFK,"New York, NY",NY,36,New York,22,12892,1289208,32575,LAX,"Los Angeles, CA",CA,6,California,91,700,657,-3.00,0,0,-1,0700-0759,19,716,1003,9,1029,1012,-17,0,0,-2,1000-1059,0,NaN,0,N,389,375,347,1,2475,10,<NA>,<NA>,<NA>,<NA>,<NA>,0
1,2022,1,1,1,6,1/1/2022 12:00:00 AM,AA,AA,19805,AA,10,AA,19805,AA,N101NN,10,12892,1289208,32575,LAX,"Los Angeles, CA",CA,6,California,91,12478,1247805,31703,JFK,"New York, NY",NY,36,New York,22,900,857,-3.00,0,0,-1,0900-0959,20,917,1647,16,1724,1703,-21,0,0,-2,1700-1759,0,NaN,0,N,324,306,270,1,2475,10,<NA>,<NA>,<NA>,<NA>,<NA>,0
2,2022,1,1,1,6,1/1/2022 12:00:00 AM,AA,AA,19805,AA,1003,AA,19805,AA,N830NN,1003,15016,1501606,31123,STL,"St. Louis, MO",MO,29,Missouri,64,13930,1393007,30977,ORD,"Chicago, IL",IL,17,Illinois,41,705,<NA>,NaN,<NA>,<NA>,<NA>,0700-0759,<NA>,<NA>,<NA>,<NA>,831,<NA>,<NA>,<NA>,<NA>,<NA>,0800-0859,1,B,0,N,86,<NA>,<NA>,1,258,2,<NA>,<NA>,<NA>,<NA>,<NA>,0
3,2022,1,1,1,6,1/1/2022 12:00:00 AM,AA,AA,19805,AA,1004,AA,19805,AA,N805NN,1004,13930,1393007,30977,ORD,"Chicago, IL",IL,17,Illinois,41,11697,1169706,32467,FLL,"Fort Lauderdale, FL",FL,12,Florida,33,1830,937,907.00,907,1,12,1800-1859,23,1000,1346,3,2239,1349,910,910,1,12,2200-2259,0,NaN,0,N,189,192,166,1,1182,5,907,0,3,0,0,0
4,2022,1,1,1,6,1/1/2022 12:00:00 AM,AA,AA,19805,AA,1007,AA,19805,AA,N870NN,1007,11057,1105703,31057,CLT,"Charlotte, NC",NC,37,North Carolina,36,15016,1501606,31123,STL,"St. Louis, MO",MO,29,Missouri,64,2240,2300,20.00,20,1,1,2200-2259,16,2316,2400,4,2345,4,19,19,1,1,2300-2359,0,NaN,0,N,125,124,104,1,575,3,16,0,0,0,3,0


In [4]:
df.tail()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,MKT_UNIQUE_CARRIER,BRANDED_CODE_SHARE,MKT_CARRIER_AIRLINE_ID,MKT_CARRIER,MKT_CARRIER_FL_NUM,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,OP_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_FIPS,ORIGIN_STATE_NM,ORIGIN_WAC,DEST_AIRPORT_ID,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,DEST,DEST_CITY_NAME,DEST_STATE_ABR,DEST_STATE_FIPS,DEST_STATE_NM,DEST_WAC,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,DEP_DELAY_GROUP,DEP_TIME_BLK,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,ARR_DELAY_GROUP,ARR_TIME_BLK,CANCELLED,CANCELLATION_CODE,DIVERTED,DUP,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,FLIGHTS,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_AIRPORT_LANDINGS
32957866,2026,3,8,31,1,8/31/2026 12:00:00 AM,NaN,NaN,<NA>,NaN,<NA>,YX,20452,YX,N979RP,4351,10994,1099402,30994,CHS,"Charleston, SC",SC,45,South Carolina,37,12953,1295304,31703,LGA,"New York, NY",NY,36,New York,22,1751,1748,-3.0,0,0,-1,1700-1759,15,1803,1932,6,2000,1938,-22,0,0,-2,2000-2059,0,NaN,0,NaN,129,110,89,1,641,3,<NA>,<NA>,<NA>,<NA>,<NA>,0
32957867,2026,3,8,31,1,8/31/2026 12:00:00 AM,NaN,NaN,<NA>,NaN,<NA>,YX,20452,YX,N979RP,4567,12953,1295304,31703,LGA,"New York, NY",NY,36,New York,22,10994,1099402,30994,CHS,"Charleston, SC",SC,45,South Carolina,37,1459,1450,-9.0,0,0,-1,1400-1459,38,1528,1710,9,1719,1719,0,0,0,0,1700-1759,0,NaN,0,NaN,140,149,102,1,641,3,<NA>,<NA>,<NA>,<NA>,<NA>,0
32957868,2026,3,8,31,1,8/31/2026 12:00:00 AM,NaN,NaN,<NA>,NaN,<NA>,YX,20452,YX,N979RP,4602,12953,1295304,31703,LGA,"New York, NY",NY,36,New York,22,10994,1099402,30994,CHS,"Charleston, SC",SC,45,South Carolina,37,905,910,5.0,5,0,0,0900-0959,29,939,1110,4,1126,1114,-12,0,0,-1,1100-1159,0,NaN,0,NaN,141,124,91,1,641,3,<NA>,<NA>,<NA>,<NA>,<NA>,0
32957869,2026,3,8,31,1,8/31/2026 12:00:00 AM,NaN,NaN,<NA>,NaN,<NA>,YX,20452,YX,N979RP,4664,10994,1099402,30994,CHS,"Charleston, SC",SC,45,South Carolina,37,12953,1295304,31703,LGA,"New York, NY",NY,36,New York,22,1200,1158,-2.0,0,0,-1,1200-1259,13,1211,1349,6,1408,1355,-13,0,0,-1,1400-1459,0,NaN,0,NaN,128,117,98,1,641,3,<NA>,<NA>,<NA>,<NA>,<NA>,0
32957870,2026,3,8,31,1,8/31/2026 12:00:00 AM,NaN,NaN,<NA>,NaN,<NA>,YX,20452,YX,N979RP,4809,10397,1039708,30397,ATL,"Atlanta, GA",GA,13,Georgia,34,12953,1295304,31703,LGA,"New York, NY",NY,36,New York,22,609,603,-6.0,0,0,-1,0600-0659,16,619,804,8,825,812,-13,0,0,-1,0800-0859,0,NaN,0,NaN,136,129,105,1,762,4,<NA>,<NA>,<NA>,<NA>,<NA>,0


In [5]:
df.shape

(32957871, 68)

In [11]:
df.describe().style.set_sticky(axis="index")

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,MKT_CARRIER_AIRLINE_ID,MKT_CARRIER_FL_NUM,OP_CARRIER_AIRLINE_ID,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,ORIGIN_STATE_FIPS,ORIGIN_WAC,DEST_AIRPORT_ID,DEST_AIRPORT_SEQ_ID,DEST_CITY_MARKET_ID,DEST_STATE_FIPS,DEST_WAC,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,DEP_DELAY_GROUP,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,ARR_DELAY_GROUP,CANCELLED,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,FLIGHTS,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,DIV_AIRPORT_LANDINGS
count,32957871.000000,32957871.000000,32957871.000000,32957871.000000,32957871.000000,11845962.000000,11845962.000000,32957871.000000,32957870.000000,32957871.000000,32957871.000000,32957871.000000,32957871.000000,32957871.000000,32957871.000000,32957871.000000,32957871.000000,32957871.000000,32957871.000000,32957871.000000,32392215.000000,32390910.000000,32390910.000000,32390910.000000,32390910.000000,32374792.000000,32374792.000000,32362463.000000,32362463.000000,32957871.000000,32362469.000000,32284610.000000,32284610.000000,32284610.000000,32284610.000000,32957871.000000,32957871.000000,32957861.000000,32284610.000000,32284610.000000,32957871.000000,32957871.000000,32957871.000000,6941163.000000,6941163.000000,6941163.000000,6941163.000000,6941163.000000,32957863.000000
mean,2023.857307,2.430481,6.273016,15.754513,3.990561,19828.266721,2474.739511,19956.779346,2492.955654,12662.609757,1266265.041996,31739.837376,27.087638,54.758899,12662.579057,1266261.971984,31739.834060,27.087552,54.758564,1327.753833,1331.506906,13.142225,16.389741,0.214261,0.204403,17.933889,1354.340872,1456.646098,8.340598,1490.572334,1459.792367,7.609666,16.295290,0.214999,-0.072283,0.017793,0.002635,145.998520,140.599632,114.340576,1.000000,826.295954,3.777731,24.945892,4.218339,13.680416,0.123574,28.112539,0.003607
std,1.354137,1.085856,3.330158,8.781800,2.007302,281.673957,1699.935652,371.705335,1656.588743,1530.925846,153092.393942,1322.726720,16.755187,26.628200,1530.926429,153092.452216,1322.711541,16.755083,26.628289,492.570057,509.388840,54.072216,55.402941,0.410309,2.395796,9.981859,511.886269,541.397807,7.003157,519.641475,546.925510,58.423693,55.085893,0.410822,2.549529,0.132197,0.051266,72.575472,72.677721,70.652747,0.000000,597.943819,2.345264,76.131305,33.745399,33.268729,3.448187,61.600948,0.106017
min,2022.000000,1.000000,1.000000,1.000000,1.000000,19393.000000,1.000000,19393.000000,1.000000,10135.000000,1013506.000000,30070.000000,1.000000,1.000000,10135.000000,1013506.000000,30070.000000,1.000000,1.000000,1.000000,1.000000,-115.000000,0.000000,0.000000,-2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,-128.000000,0.000000,0.000000,-2.000000,0.000000,0.000000,-272.000000,14.000000,5.000000,1.000000,11.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2023.000000,1.000000,3.000000,8.000000,2.000000,19790.000000,1083.000000,19790.000000,1146.000000,11292.000000,1129202.000000,30647.000000,12.000000,34.000000,11292.000000,1129202.000000,30647.000000,12.000000,34.000000,907.000000,909.000000,-6.000000,0.000000,0.000000,-1.000000,12.000000,926.000000,1040.000000,4.000000,1102.000000,1043.000000,-15.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,92.000000,87.000000,63.000000,1.000000,391.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2024.000000,2.000000,6.000000,16.000000,4.000000,19805.000000,2158.000000,19930.000000,2213.000000,12889.000000,1288904.000000,31453.000000,26.000000,45.000000,12889.000000,1288904.000000,31453.000000,26.000000,45.000000,1320.000000,1324.000000,-2.000000,0.000000,0.000000,-1.000000,15.000000,1338.000000,1458.000000,6.000000,1516.000000,1502.000000,-6.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,129.000000,124.000000,97.000000,1.000000,674.000000,3.000000,4.000000,0.000000,0

## Functions

In [26]:
"""
basically print the average of all the flights form the year 2022 till 2026 
single average is the output for all the flights, (basically 32957871 / 12)

"""



def calculate_avg_flights_per_month(df):
    count = len(df["MONTH"])
    result = count/12
    return result

avg = calculate_avg_flight_per_month(df)
print(avg)


2746489.25


In [13]:
count = len(df["MONTH"])
print(count)

32957871


In [19]:
df["MONTH"].value_counts().sort_index()

MONTH
1     2768635
2     2595323
3     3011517
4     2941040
5     3046706
6     3045918
7     3155796
8     3083677
9     2294790
10    2415631
11    2277238
12    2321600
Name: count, dtype: int64

## Functions using Groupby

In [41]:

"""
Q . calculates the average number of flight records per year

Hint for groupby syntax is "df.groupby("Categorical_Column")["Numerical_Column"].operation()"
CATEGORICAL col FIRST and then the NUMERICAL cols comes.
"""



def avg_flights_per_year(df):
    yearly_flights = df.groupby("YEAR").size()
    avg = yearly_flights.mean()
    return avg


avg = avg_flights_per_year(df)
print(avg)
print(type(yearly_flights))                                
    

print(f" We have {avg} flights per year")
    

6591574.2
<class 'pandas.core.series.Series'>
 We have 6591574.2 flights per year


In [37]:
yearly_flights = df.groupby("YEAR").size()
print(yearly_flights)

YEAR
2022    7013508
2023    7134931
2024    7079081
2025    7001619
2026    4728732
dtype: int64


In [46]:
"""
Q. average number of flight records per month across all years

"""


def avg_flights_per_month_across_all_years(df):
    month = df.groupby("MONTH").size()
    avg = month.mean()
    return avg

avg = avg_flights_per_month_across_all_years(df)
print(avg)

2746489.25


In [47]:
a=df.groupby("MONTH").size()
print(a)

MONTH
1     2768635
2     2595323
3     3011517
4     2941040
5     3046706
6     3045918
7     3155796
8     3083677
9     2294790
10    2415631
11    2277238
12    2321600
dtype: int64


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32957871 entries, 0 to 32957870
Data columns (total 68 columns):
 #   Column                  Dtype   
---  ------                  -----   
 0   YEAR                    int16   
 1   QUARTER                 int8    
 2   MONTH                   int8    
 3   DAY_OF_MONTH            int8    
 4   DAY_OF_WEEK             int8    
 5   FL_DATE                 category
 6   MKT_UNIQUE_CARRIER      category
 7   BRANDED_CODE_SHARE      category
 8   MKT_CARRIER_AIRLINE_ID  UInt16  
 9   MKT_CARRIER             category
 10  MKT_CARRIER_FL_NUM      Int16   
 11  OP_UNIQUE_CARRIER       category
 12  OP_CARRIER_AIRLINE_ID   int16   
 13  OP_CARRIER              category
 14  TAIL_NUM                category
 15  OP_CARRIER_FL_NUM       Int16   
 16  ORIGIN_AIRPORT_ID       int16   
 17  ORIGIN_AIRPORT_SEQ_ID   int32   
 18  ORIGIN_CITY_MARKET_ID   uint16  
 19  ORIGIN                  category
 20  ORIGIN_CITY_NAME        category
 21  ORIGIN

In [10]:
df.columns

Index(['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'MKT_UNIQUE_CARRIER', 'BRANDED_CODE_SHARE', 'MKT_CARRIER_AIRLINE_ID', 'MKT_CARRIER', 'MKT_CARRIER_FL_NUM', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID', 'OP_CARRIER',
       'TAIL_NUM', 'OP_CARRIER_FL_NUM', 'ORIGIN_AIRPORT_ID', 'ORIGIN_AIRPORT_SEQ_ID', 'ORIGIN_CITY_MARKET_ID', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'ORIGIN_STATE_FIPS', 'ORIGIN_STATE_NM', 'ORIGIN_WAC', 'DEST_AIRPORT_ID',
       'DEST_AIRPORT_SEQ_ID', 'DEST_CITY_MARKET_ID', 'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR', 'DEST_STATE_FIPS', 'DEST_STATE_NM', 'DEST_WAC', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15', 'DEP_DELAY_GROUP',
       'DEP_TIME_BLK', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15', 'ARR_DELAY_GROUP', 'ARR_TIME_BLK', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'DUP',
       'CRS_ELAPSED_TIME', 'ACTUAL_ELAPSED_TIME', 'AIR_